# Letting the model drive

MichAl Academy, unit 5.4.

Run each cell with **Shift+Enter**.

Unit 5.3 built a fixed pipeline: take the question, search, put what comes back
in the prompt, answer. The code decided when to search and what to search for.

An **agent** moves that decision to the model. It writes down which tool it
wants and what to pass it, your code runs the tool and hands back the result,
and the model goes again. The loop runs until it answers or you stop it.

This notebook builds that loop on unit 5.3's corpus, so every number here is
comparable with a number there, and measures what the loop costs as well as
what it buys.


In [ ]:
import re
import ast
import warnings
from collections import Counter

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

warnings.filterwarnings("ignore")
torch.set_num_threads(1)

NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(NAME)
model = AutoModelForCausalLM.from_pretrained(NAME)
model.eval()


def chat(messages):
    return tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)


@torch.no_grad()
def generate(prompt, n=40):
    ids = tok(prompt, return_tensors="pt").input_ids
    out = model.generate(ids, max_new_tokens=n, do_sample=False,
                         pad_token_id=tok.eos_token_id,
                         attention_mask=torch.ones_like(ids))
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True).strip()


## The corpus and the tool

The same twenty-four invented sentences as unit 5.3, and the same twelve
questions. Nothing here is in any model's training data, so a right answer can
only have come through the tool.


In [ ]:
PASSAGES = [
    "The Kestrel KX-7712 appliance ships with 48 gigabytes of memory and two power supplies.",
    "The Kestrel KX-7712 draws 310 watts under full load and 95 watts at idle.",
    "Firmware branch 9.4 for the Kestrel line adds support for the Harrier telemetry format.",
    "Firmware branch 9.4 removes the legacy Osprey configuration importer.",
    "The Kestrel KX-3300 is the desk-side model and has a single power supply.",
    "The Kestrel KX-3300 holds 16 gigabytes of memory and cannot be expanded.",
    "Kestrel appliances listen for management traffic on TCP port 8443 by default.",
    "The management port on a Kestrel can be moved, but not to any port below 1024.",
    "A Kestrel cluster requires an odd number of members, and the supported sizes are three, five and seven.",
    "A Kestrel cluster loses write availability when more than half its members are unreachable.",
    "The Harrier telemetry format writes one record per flow and compresses at roughly four to one.",
    "Harrier records carry a sixteen byte flow identifier and a millisecond timestamp.",
    "The Osprey configuration importer was deprecated in firmware 9.2 and removed in 9.4.",
    "Configurations exported by Osprey can be converted with the kestrel-convert utility.",
    "The kestrel-convert utility refuses to run on a configuration larger than 64 megabytes.",
    "Support contract tier Gold includes four hour hardware replacement on the Kestrel line.",
    "Support contract tier Silver includes next business day hardware replacement.",
    "The Kestrel warranty covers power supplies for five years and fans for three.",
    "Kestrel appliances log to a ring buffer of 2 gigabytes before overwriting the oldest entries.",
    "Setting the log level to verbose on a Kestrel fills the ring buffer in about six hours.",
    "The Kestrel KX-7712 was released in March 2024 and the KX-3300 followed in September 2024.",
    "Kestrel firmware branch 9.3 is supported until the end of 2026.",
    "A Kestrel appliance performs a configuration backup every night at 02:00 local time.",
    "Kestrel nightly backups are kept for thirty days unless a retention policy says otherwise.",
]

QUESTIONS = [
    ("How much memory does the Kestrel KX-7712 have?", "48", 0),
    ("How many watts does a KX-7712 draw under full load?", "310", 1),
    ("Which telemetry format does firmware 9.4 add support for?", "harrier", 2),
    ("How many power supplies does the KX-3300 have?", "one", 4),
    ("Which TCP port do Kestrel appliances use for management?", "8443", 6),
    ("What cluster sizes are supported?", "three", 8),
    ("How big is the flow identifier in a Harrier record?", "sixteen", 11),
    ("In which firmware version was the Osprey importer removed?", "9.4", 12),
    ("What is the size limit for kestrel-convert?", "64", 14),
    ("What replacement time does the Gold tier include?", "four", 15),
    ("How long are power supplies covered by the warranty?", "five", 17),
    ("At what time does a Kestrel run its nightly backup?", "02:00", 22),
]


def tokenise(text):
    return re.findall(r"[a-z0-9.:-]+", text.lower())


def bm25_factory(docs):
    """Unit 5.3.4's scorer, unchanged."""
    toks = [tokenise(d) for d in docs]
    df = Counter(w for d in toks for w in set(d))
    avgdl = sum(len(d) for d in toks) / len(toks)
    n = len(toks)

    def score(q, k1=1.5, b=0.75):
        out = np.zeros(n)
        for w in tokenise(q):
            if w not in df:
                continue
            idf = np.log(1 + (n - df[w] + 0.5) / (df[w] + 0.5))
            for i, d in enumerate(toks):
                f = d.count(w)
                if f:
                    out[i] += idf * f * (k1 + 1) / (f + k1 * (1 - b + b * len(d) / avgdl))
        return out
    return score


bm25 = bm25_factory(PASSAGES)


def search_index(query):
    return int(bm25(query).argmax())


def search(query):
    """The tool. One string in, one passage out."""
    return PASSAGES[search_index(query)]


print(search("management port"))


## The pipeline with no loop

Unit 5.3's arrangement, repeated here so the rest of the notebook has something
to be compared against. The query is the question itself, there is exactly one
search, and the model is asked once.


In [ ]:
def answer_with(question, note=None):
    if note is None:
        msgs = [{"role": "user", "content": question}]
    else:
        msgs = [{"role": "user", "content":
                 f"Answer the question using only the note below.\n\n"
                 f"Note: {note}\n\nQuestion: {question}"}]
    return generate(chat(msgs), n=24)


closed = fixed = fixed_hit = 0
for q, want, gold in QUESTIONS:
    closed += want.lower() in answer_with(q).lower()
    i = search_index(q)
    fixed_hit += i == gold
    fixed += want.lower() in answer_with(q, PASSAGES[i]).lower()

print(f"no search at all              {closed:>3}/12")
print(f"one search, question as query  found {fixed_hit}/12   answered {fixed}/12")


**0 and 9**, the same two numbers unit 5.3 measured. Everything below has to
beat 9 to be worth its extra calls.


## The loop, written out

One function. The model is told the format, we read its reply, and if it asked
for the tool we run the tool and put the result back in the conversation as a
turn.

The parser is deliberately blunt: find `SEARCH:` or `ANSWER:` anywhere in the
reply and take the rest of the line. Being generous about what counts as an
action matters, because a small model's formatting is not reliable.


In [ ]:
RULE = (
    "You can look things up in the Kestrel manual.\n"
    "To look something up, reply with one line:\n"
    "SEARCH: the words to look up\n"
    "When you know the answer, reply with one line:\n"
    "ANSWER: the answer\n"
)


def parse_action(text):
    m = re.search(r"\bSEARCH:\s*(.+)", text)
    if m:
        return "search", m.group(1).strip().splitlines()[0][:120]
    m = re.search(r"\bANSWER:\s*(.+)", text)
    if m:
        return "answer", m.group(1).strip().splitlines()[0][:120]
    return "none", text.strip()[:120]


def run(question, opening, cap=8, show=False, stop_on_repeat=False):
    """opening is the list of messages the conversation starts with.
    Returns the final answer or None, the trace, and the prompt tokens read.
    With stop_on_repeat, the loop also gives up as soon as the model asks for
    exactly the same thing twice in a row."""
    msgs = opening + [{"role": "user", "content": f"Question: {question}"}]
    trace, spent = [], 0
    for _ in range(cap):
        prompt = chat(msgs)
        spent += tok(prompt, return_tensors="pt").input_ids.shape[1]
        kind, payload = parse_action(generate(prompt))
        if stop_on_repeat and trace and (kind, payload) == trace[-1]:
            return None, trace, spent
        trace.append((kind, payload))
        if show:
            print(f"    {kind:<7} {payload}")
        if kind == "answer":
            return payload, trace, spent
        if kind == "search":
            note = search(payload)
            if show:
                print(f"    result  {note}")
            msgs += [{"role": "assistant", "content": f"SEARCH: {payload}"},
                     {"role": "user", "content": f"RESULT: {note}"}]
        else:
            msgs += [{"role": "assistant", "content": payload},
                     {"role": "user", "content":
                      "Reply with one SEARCH line or one ANSWER line."}]
    return None, trace, spent


BARE = [{"role": "user", "content": RULE}]
print(QUESTIONS[4][0])
run(QUESTIONS[4][0], BARE, cap=4, show=True)


Read that trace. The model answered from nothing on the first move, then asked to
search for **"the words to look up"**, which is the placeholder out of the rule
copied back verbatim, and then wrote its own `RESULT:` line rather than waiting
for ours.

None of that is a bug in the loop. The loop is fine; the model never entered it.


## Demonstrating the format instead of describing it

Unit 5.1.2 measured this exact difference on the classification task: examples
pasted into a message were worth nothing, and the same examples as conversation
turns were worth 21 of 24 on formatting. The same move applies to an action
format. Show the model one complete turn of the loop, as real turns.


In [ ]:
DEMO = [
    {"role": "user", "content": RULE + "\nQuestion: How long does the warranty cover fans?"},
    {"role": "assistant", "content": "SEARCH: warranty fans"},
    {"role": "user", "content": "RESULT: The Kestrel warranty covers power supplies for five years and fans for three."},
    {"role": "assistant", "content": "ANSWER: three years"},
]

for q in (QUESTIONS[2][0], QUESTIONS[1][0]):
    print(q)
    run(q, DEMO, cap=4, show=True)
    print()


Two runs, and the loop worked mechanically in both. One search each, a result
each, an answer each, no cap reached.

The first is the thing working: a query of its own choosing, the right sentence
back, the answer read out of it.

The second picks an equally sensible query and gets the right sentence too. That
sentence holds two numbers, 310 watts under full load and 95 at idle, and the
question asked for the first. The answer is 95. Nothing failed that a
`try`/`except` could catch.


## What the loop is worth

Three columns, because the loop can fail in three places: it can choose a bad
query, it can misread what came back, and it can fail to stop. Score them apart.


In [ ]:
found = answered = ended = 0
retrieved = []
for q, want, gold in QUESTIONS:
    final, trace, _ = run(q, DEMO, cap=4)
    got = [search_index(p) for k, p in trace if k == "search"]
    queries = [p for k, p in trace if k == "search"]
    retrieved.append(got[-1] if got else None)
    found += gold in got
    ended += final is not None
    right = final is not None and want.lower() in final.lower()
    answered += right
    print(f"{'right' if right else '     '} {'found' if gold in got else '     '}  "
          f"query {str(queries):<42} -> {final}")

print()
print(f"the loop:  found {found}/12   answered {answered}/12   stopped on its own {ended}/12")

# The control. Same passages the loop chose, read in one plain call instead of
# inside the conversation: does the reading get worse in the loop, or was the
# passage simply not good enough?
control = 0
for (q, want, gold), i in zip(QUESTIONS, retrieved):
    if i is not None:
        control += want.lower() in answer_with(q, PASSAGES[i]).lower()
print(f"the same passages, answered in one call:     {control}/12")


**The loop ran perfectly and answered 2 of 12.** It stopped on its own every
time, which is the part of the machinery under our control, and it lost at both
of the steps that are not.

**Its queries were worse than the question.** Using the question itself as the
query found the right passage 10 times; the queries the model wrote found it 5.
Read the query column: three of the twelve are `the words to look up` or `the
words to search`, the placeholder out of the rule handed back as if it were a
search term, and one is `port 123`, a number that appears nowhere in the corpus
or the question.

**Then it misread most of what it did find.** Five gold passages came back and
two questions were answered.

The control line separates the loop from the reading. Those same passages,
handed over one at a time in a plain prompt with no conversation around them,
answered 4 rather than 2. That is two questions out of twelve and too small to
lean on, but it points the same way as everything else here: the extra turns
are not free.

**Nothing in this task needed a second step.** One question, one lookup, one
answer, which is exactly the shape unit 5.3 built a fixed pipeline for, and the
fixed pipeline answers 9. Anthropic's advice for production systems is to find
the simplest arrangement that works and add machinery only when it demonstrably
helps. Here the loop is machinery the task did not ask for, and it costs seven
answers.


## Two tools

A loop earns its cost when there is a real choice to make. Add a calculator and
mix in six sums, and the model has to decide which tool a question needs.


In [ ]:
def calc(expr):
    """Arithmetic only. The model wrote this string, so it is parsed into a
    syntax tree and walked rather than executed."""
    ops = {ast.Add: lambda a, b: a + b, ast.Sub: lambda a, b: a - b,
           ast.Mult: lambda a, b: a * b, ast.Div: lambda a, b: a / b}

    def walk(node):
        if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
            return node.value
        if isinstance(node, ast.BinOp) and type(node.op) in ops:
            return ops[type(node.op)](walk(node.left), walk(node.right))
        if isinstance(node, ast.UnaryOp) and isinstance(node.op, ast.USub):
            return -walk(node.operand)
        raise ValueError("not arithmetic")

    try:
        return str(round(walk(ast.parse(expr.strip(), mode="eval").body), 4))
    except Exception:
        return "error"


TWO_RULE = (
    "You have two tools.\n"
    "SEARCH: words   looks something up in the Kestrel manual.\n"
    "CALC: sum       works out an arithmetic sum.\n"
)

TWO_DEMO = [
    {"role": "user", "content": TWO_RULE + "\nQuestion: How long does the warranty cover fans?"},
    {"role": "assistant", "content": "SEARCH: warranty fans"},
    {"role": "user", "content": "RESULT: The Kestrel warranty covers power supplies for five years and fans for three."},
    {"role": "assistant", "content": "ANSWER: three years"},
    {"role": "user", "content": "Question: What is 12 plus 7?"},
    {"role": "assistant", "content": "CALC: 12 + 7"},
    {"role": "user", "content": "RESULT: 19"},
    {"role": "assistant", "content": "ANSWER: 19"},
]

SUMS = [
    ("What is 310 plus 95?", "405"),
    ("What is 48 times 3?", "144"),
    ("What is 64 minus 16?", "48"),
    ("What is 2 times 30?", "60"),
    ("What is 8443 minus 1024?", "7419"),
    ("What is 16 plus 48?", "64"),
]


def parse_two(text):
    m = re.search(r"\bCALC:\s*(.+)", text)
    if m:
        return "calc", m.group(1).strip().splitlines()[0][:80]
    return parse_action(text)


def run_two(question, cap=4):
    msgs = list(TWO_DEMO) + [{"role": "user", "content": f"Question: {question}"}]
    first = None
    for _ in range(cap):
        kind, payload = parse_two(generate(chat(msgs)))
        if first is None and kind in ("calc", "search"):
            first = kind
        if kind == "answer":
            return first, payload
        if kind in ("calc", "search"):
            result = calc(payload) if kind == "calc" else search(payload)
            msgs += [{"role": "assistant", "content": f"{kind.upper()}: {payload}"},
                     {"role": "user", "content": f"RESULT: {result}"}]
        else:
            msgs += [{"role": "assistant", "content": payload},
                     {"role": "user", "content": "Reply with one tool line or one ANSWER line."}]
    return first, None


In [ ]:
picked = ok = 0
for q, want, _ in QUESTIONS[:6]:
    first, final = run_two(q)
    picked += first == "search"
    ok += final is not None and want.lower() in final.lower()
print(f"six lookups:  reached for search {picked}/6   answered {ok}/6")

picked = ok = 0
for q, want in SUMS:
    first, final = run_two(q)
    picked += first == "calc"
    ok += final is not None and want in final
print(f"six sums:     reached for calc   {picked}/6   answered {ok}/6")

unaided = sum(w in generate(chat([{"role": "user", "content": q}]), n=16) for q, w in SUMS)
print(f"the same sums with no tool offered:          answered {unaided}/6")


**The sums go 6 for 6 on both counts, against 1 of 6 unaided.** This is the same
thing unit 5.2.3 measured from a different angle: the tool does not make the model
cleverer, it moves the arithmetic to something that was already right.

**The lookups reach for the calculator half the time.** A question with digits and
the word "plus" in it advertises which tool it needs. "How many power supplies does
the KX-3300 have?" does not: choosing to search requires knowing that you do not
know, and nothing in the question's surface says so.

That asymmetry is the practical content of tool choice. Tools whose trigger is
visible in the request get picked. Tools that have to be reasoned for do not, and
every tool you add is another wrong option.


## Loops that do not stop

Every measurement above ran under a cap. Take the cap seriously for a moment and
look at what it is protecting you from: the same loop with and without the
demonstration, allowed eight steps each.


In [ ]:
print(f"{'':<24}{'hit the cap':>12}{'repeated itself':>17}{'mean steps':>12}{'tokens read':>13}")
for label, opening in (("no demonstration", BARE), ("with a demonstration", DEMO)):
    capped = repeats = 0
    steps, spends = [], []
    for q, _, _ in QUESTIONS:
        final, trace, spent = run(q, opening, cap=8)
        capped += final is None
        actions = [f"{k}:{p}" for k, p in trace]
        repeats += any(a == b for a, b in zip(actions, actions[1:]))
        steps.append(len(trace))
        spends.append(spent)
    print(f"{label:<24}{capped:>9}/12{repeats:>14}/12"
          f"{np.mean(steps):>12.1f}{int(np.mean(spends)):>13}")


**Twelve of twelve against none.** The model that never entered the loop does not
sit there quietly. It runs to the cap on every question, asks for the same thing
twice in a row on eleven of them, and reads 2,184 prompt tokens per question to
produce no answer at all.

That is the failure mode to design for. Nothing raises, nothing times out, every
call returns a perfectly ordinary reply, and the bill is five times the working
loop's for twelve answers fewer. A loop with no cap would still be running.


## Stopping on a repeat

The cap is the crude guard. A cheaper one is to notice that the model has asked
for exactly the same thing twice in a row, because a loop repeating itself has no
new information to work with and the next step cannot differ.


In [ ]:
stopped = capped = answered = 0
spends = []
for q, want, _ in QUESTIONS:
    final, trace, spent = run(q, BARE, cap=8, stop_on_repeat=True)
    hit_cap = final is None and len(trace) == 8
    capped += hit_cap
    stopped += final is None and not hit_cap
    answered += final is not None
    spends.append(spent)

print(f"stopped on a repeat {stopped}/12   ran to the cap {capped}/12   "
      f"answered {answered}/12   mean tokens read {int(np.mean(spends))}")


**322 tokens against 2,184.** The detector answers nothing extra. It notices
sooner that nothing more is coming, which is all a stopping condition is for.

Keep the cap anyway. A model can wander between two different useless actions
forever without ever repeating one exactly, and the repeat test never fires on
that; here it happened to catch all twelve, on a different corpus it will catch
some. The cap is the guard that cannot be evaded.


## What to take from this

**The loop is small.** Read the model's reply, run what it asked for, put the
result back, repeat. There is no more to an agent than that, and every framework
you meet is this loop with logging and retries around it.

**Score the steps separately.** Query choice, reading and stopping fail
independently and have different fixes. The loop here stopped 12 times out of 12,
found 5 and answered 2, and a single end-to-end number would have told you none
of that.

**Ask whether the task needs a loop at all.** Twelve one-step questions did not:
unit 5.3's fixed pipeline answers 9 where the loop answers 2, for fewer tokens
and no chance of running away.

**A tool is worth most where the model is worst.** The same loop that lost seven
answers on lookups turned 1 of 6 into 6 of 6 on arithmetic.

**Cap it, and watch for repeats.** Both, not either.
